<img src="../shared/alchemi-banner-left.png" alt="NVIDIA ALCHEMI: AI for Chemistry and Materials Science" style="display:block;box-sizing:border-box;width:100%;max-width:100%;height:auto;">

<img src="assets/agent07-top-light.svg" alt="In one GPU batch each trajectory moves from NVT to NVE on its own; beside it a queue feeds two batches of four slots: in the static batch finished slots sit idle until the slowest system is done, in the inflight batch each finished slot takes the next system at once, so the batch slots stay in use." style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0;">

<div style="border: 1px solid rgba(127,127,127,0.35); border-left: 5px solid #76b900; border-radius: 4px; padding: 24px 28px; margin: 8px 0 18px; background: transparent; color: inherit; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif;"><div style="font-variant: small-caps; font-size: 12px; font-weight: 700; letter-spacing: 2px; opacity: 0.75;">AGENTIC TRACK, NOTEBOOK 07</div><h1 style="margin: 6px 0 8px; padding: 0; border: 0;"><span style="font-size: 32px; font-weight: 700; line-height: 1.2; color: inherit;">One out, one in</span></h1><div style="font-size: 18px; line-height: 1.45; opacity: 0.85;">Run a multi-stage MD study on thousands of trajectories and keep the GPU full from the first step to the last.</div></div>

<div style="border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 12px 18px; margin: 12px 0 18px; background: transparent; color: inherit; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; line-height: 1.6;"><b>Hardware:</b> a CUDA-capable NVIDIA GPU<br><b>Prerequisites:</b> Meet your agent; Made to order; Hooked in at every step<br><b>References:</b> <a href="https://nvidia.github.io/nvalchemi-toolkit/" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Toolkit documentation</a>; <a href="https://nvidia.github.io/nvalchemi-toolkit/examples/" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Toolkit examples</a>; <a href="https://github.com/NVIDIA/skills/tree/main/skills/nvalchemi-toolkit" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">the nvalchemi-toolkit skill</a></div>

<img src="assets/agent07-roadmap-light.svg" alt="Sections of this notebook on one line." style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0;">

In [ ]:
# Load PyTorch and the course helpers, then choose the GPU
import torch
from helpers import agent07 as helpers

helpers.configure_tutorial()             # course display defaults
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In a batch, a trajectory that is done still holds its place until the
slowest one finishes. On thousands of systems that idle space is about half
of the GPU. Fused stages let each trajectory move on when it is ready,
and that independence is what refill builds on: a finished trajectory
hands its place to the next one waiting. You will build both, and the
staged run they replace.

In [ ]:
# Play the scheduling animation
helpers.show_scheduling_widget()

In [ ]:
# Prepare the study, set the capacity, and print what your agent needs
batch_multiplier = 26  # trajectories per complex: 26 gives 3,874
# The capacity: a batch size at which this GPU is close to fully busy
# (about 90 % of its top throughput); above it a step takes longer almost
# in proportion to the atoms.
max_batch_size = 1308  # trajectories in the active batch
max_atoms = 20000      # atoms in the active batch
# NVT ends once the mean temperature over the last `window` MD steps is within `tolerance` K
# of `temperature`, or at `nvt_steps`; NVE then runs `nve_steps` MD steps. `dt` is the timestep
# in fs, `friction` the Langevin friction per fs.
stage_settings = dict(temperature=300.0, tolerance=60.0, window=100, nvt_steps=600, nve_steps=200, dt=1.0, friction=0.01)
trajectories, composed_model = helpers.prepare_study(
    device, batch_multiplier=batch_multiplier)
work_dir = helpers.make_work_dir()       # notebooks/agent07_work
helpers.show_session()                   # prints what your agent needs to start

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">1. Two stages in one batch</h2>

#### Example: fuse NVT and NVE

Each trajectory starts in NVT at 300 K and moves to NVE once its
temperature has settled; NVE then runs a fixed number of steps.
Fused stages keep both in one batch: at every step each trajectory gets the
update of the stage it is in.

**Joining stages**

```python
pipeline = nvt + nve
pipeline = FusedStage(
    sub_stages=[(0, nvt), (1, nve)])
```

Both lines build the same pipeline. The written-out form also takes the
parts refill needs.

| | |
| --- | --- |
| input  | `trajectories`; `composed_model`; `stage_settings`; `max_batch_size`, `max_atoms` |
| output | `build_stages`, a function that returns a fresh NVT and NVE pair, NVT with its stopping rule; `fused_batches`, `fused_seconds`, `stage_trace`; in `agent07_work/fused.py` |

> 💡 **Hint**
>
> `helpers.show_stage_changes(stage_trace)` draws the result. `build_stages`
> keeps the NVT stopping rule per trajectory by `simulation_id`, and `stage_trace`
> holds two integers per trajectory, in `simulation_id` order: the MD step of
> the move to NVE and the MD step of the finish.

In [ ]:
%load agent07_work/fused.py

In [ ]:
# Show when your trajectories changed stage
helpers.show_stage_changes(stage_trace)

In a staged run every trajectory stays in NVT until the last one is ready.
Here the first one moved on earlier; the last line says by how many MD steps.

> 💡 **Hint**
>
> A stage with no stopping rule of its own, other than the last, moves a
> system on as soon as its forces are small. Give NVT its own rule, or some
> trajectories will leave it early.

> ⚙️ **Fine control**
>
> ```python
> NVTLangevin(
>     model=...,
>     dt=...,           # timestep, fs
>     temperature=...,  # K
>     friction=...,     # 1/fs
>     n_steps=...,      # step budget per trajectory in this stage
> )
> ```
>
> Example: `run NVT at 350 K`

> </> API
>
> `ConvergenceHook(criteria=[...])`
>
> **Input:** one or more criteria, each a field of the batch and its threshold<br>
> **Effect:** the stopping rule of a stage; per trajectory inside a fused stage

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">2. One out, one in</h2>

#### Example: refill the batch

Refill needs three places: a waiting list, the active batch, and a store
for finished results. Every few steps Toolkit moves finished trajectories
to the store and fills the freed space from the waiting list.

| | |
| --- | --- |
| input  | `trajectories`; `composed_model`; `build_stages`; `max_batch_size`, `max_atoms`; `helpers.CudaCompletionSink` |
| output | `completed`, `inflight_seconds`, in `agent07_work/refill.py` |

> 💡 **Hint**
>
> The waiting list is a dataset: `InMemoryDataset` over one `Batch` of your
> trajectories. The store goes to `FusedStage` as `sinks=[...]`, the waiting
> list as `sampler=...`.

In [ ]:
%load agent07_work/refill.py

In [ ]:
# Count the finished trajectories and the ones that entered later
helpers.show_refill(completed, trajectories)

The table counts the trajectories that finished and how many of them
entered after the first batch was full.

> ⚙️ **Fine control**
>
> ```python
> SizeAwareSampler(
>     waiting,
>     max_batch_size=...,   # trajectories in the active batch
>     max_atoms=...,        # atoms in the active batch
> )
> FusedStage(..., refill_frequency=...)   # steps between refill checks
> ```
>
> Example: `check for free places every 10 steps`

#### Try it: run it in stages

The ordinary way has no fused stages: one group runs NVT until every
trajectory has passed its rule or reached its step cap, then NVE, then the next group starts. A
trajectory that finishes early waits for its group. This is the baseline that refill is measured against.

| | |
| --- | --- |
| input  | `trajectories`; `composed_model`; `build_stages`; `max_batch_size`, `max_atoms` |
| output | `staged_batches`, `staged_seconds`, in `agent07_work/staged.py` |

Done when: every trajectory has finished both stages.

> 💡 **Hint**
>
> `trajectories` from the setup cell is the tested default: run it as it is.

In [ ]:
%load agent07_work/staged.py

In [ ]:
# Count the groups and the trajectories that finished
helpers.show_staged(staged_batches, trajectories)

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">3. Compare the three routes</h2>

Every row runs the same workflow: the same starting structures, model,
stages and capacity. Only the scheduling differs.

In [ ]:
# Put the three routes in one table
helpers.compare_routes(
    staged=(staged_batches, staged_seconds),
    fused=(fused_batches, fused_seconds),
    inflight=(completed, inflight_seconds))

"Finished" should be equal in all three rows. "Admitted inflight" counts
the trajectories that entered after the first batch was full; only the
refill route has any. Compare the times with that in mind.

Fused stages in fixed groups take about as long as the staged run, since
each group still waits for its slowest trajectory; in some studies they
save a little. Refill removes that wait.

#### Try it: relax, then MD

The pair most studies start with: relax each structure, then simulate it.
Each trajectory enters MD as soon as its own relaxation has converged.

| | |
| --- | --- |
| input  | `trajectories`; `composed_model`; `max_batch_size`, `max_atoms` |
| output | `relax_md_batches`, `relax_md_trace`, in `agent07_work/relax_md.py` |

Done when: the summary gives the earliest and the latest step at which trajectories entered MD.

> 💡 **Hint**
>
> `helpers.show_stage_changes(relax_md_trace)` draws the result.
> `relax_md_trace` has the form of `stage_trace`: two integers per
> trajectory, in `simulation_id` order, the step of the move to MD and the
> step of the finish.

> 💡 **Hint**
>
> `trajectories` from the setup cell is the tested default here too: the
> complexes as published, not yet relaxed with this model.

In [ ]:
%load agent07_work/relax_md.py

In [ ]:
# Show when your trajectories entered MD
helpers.show_stage_changes(relax_md_trace)

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">Explore more</h2>

More on stages, stores and stopping rules:

<div style="display: flex; flex-wrap: wrap; gap: 16px; margin: 4px 0 18px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif;"><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5; overflow-wrap: break-word;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div style="overflow-wrap: break-word;"><b><a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/examples/intermediate/01_multistage_pipeline.py">Three stages</a></b></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8; overflow-wrap: break-word;">Relax, thermalize and sample in one batch: <code>relax + nvt + nve</code>.</div></div><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5; overflow-wrap: break-word;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div style="overflow-wrap: break-word;"><b><a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/examples/intermediate/04_inflight_batching.py">Toolkit's own stores</a></b></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8; overflow-wrap: break-word;"><code>ConvergedSnapshotHook</code> fills a <code>HostMemory</code> sink adapted to keep <code>system_id</code>.</div></div><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5; overflow-wrap: break-word;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div style="overflow-wrap: break-word;"><b><a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/examples/advanced/03_custom_convergence.py">Your own stopping rule</a></b></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8; overflow-wrap: break-word;">Several criteria in one <code>ConvergenceHook</code>, or a function of your own.</div></div></div>

<div style="border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; background: transparent; color: inherit; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; padding: 15px 20px; margin: 24px 0 18px; line-height: 1.6;"><h2 style="margin: 0 0 10px; padding: 0; border: 0; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 24px; font-weight: 500; line-height: 1.25; color: inherit;">Notebook 07 recap</h2><div style="margin: 0 0 10px;"><code>stage_a + stage_b</code>: any stages join this way, and each system moves on at its own pace. With refill, the batch stays full while work is waiting.</div><div style="margin: 0;"><img src="assets/fork-agent-light.svg" alt="" width="18" height="18" style="width: 18px; height: 18px; vertical-align: -3px; margin-right: 6px;"><b>Next:</b> <a href="alchemi-agent-08-paths-of-least-resistance.ipynb" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Paths of least resistance, batched →</a></div></div>